# Desafio Lighthouse - Análise de Dados e IA
## Empresa: LH Nautical

**Autor:** Mauro Tessaro Jr  
**Data:** Março 2026  

### Sobre o projeto
A LH Nautical é uma empresa de varejo náutico com loja física em Florianópolis 
e e-commerce nacional. Este notebook consolida as análises do desafio técnico, 
cobrindo desde limpeza de dados até modelos preditivos e sistemas de recomendação.

### Stack utilizada
- **Python 3.11** — tratamento de dados e modelos
- **DuckDB** — queries SQL embarcadas
- **Pandas** — manipulação de DataFrames

In [1]:
import pandas as pd
import duckdb

# Caminhos dos datasets
VENDAS = "vendas_2023_2024.csv"
PRODUTOS = "produtos_raw.csv"
CUSTOS = "custos_importacao.json"
CLIENTES = "clientes_crm.json"

## Q2 - Limpeza e Normalização de Produtos

Normalização do arquivo `produtos_raw.csv`:
1. Padronizar categorias em: `eletrônicos`, `propulsão`, `ancoragem`
2. Converter valores para tipo numérico
3. Remover duplicatas

In [2]:
# Lê o csv de produtos e armazena em um dataframe:
df_produtos = pd.read_csv(PRODUTOS)

# Mostra quantas linhas e colunas tem o DataFrame (linhas, colunas)
print("número de linhas =", df_produtos.shape[0])
print("número de colunas =", df_produtos.shape[1])

# Mostra os nomes de todas as colunas
print(df_produtos.columns.tolist())

# Mostra quantas vezes cada categoria aparece — revela os nomes "sujos"
print("\n",df_produtos['actual_category'].value_counts())

número de linhas = 157
número de colunas = 4
['name', 'price', 'code', 'actual_category']

 actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
Eletronicoz              7
eletrônicos              7
ELETRONICOS              6
E L E T R Ô N I C O S    6
PROPULSAO                6
P R O P U L S Ã O        6
propulsão                6
Eletrunicos              5
eLeTrÔnIcOs              5
Propulção                5
Prop                     5
Propulssão               5
Encoragem                5
Ancorajm                 5
A N C O R A G E M        5
aNcOrAgEm                5
Eletrônicos              4
propulsao                4
eletronicos              3
EletrônicoS              3
Propulçao                3
Ancoragem                3
Ancorajem                3
Eletroniscos             2
Eletronicos              2
pRoPuLsÃo                2
Propulsam                2
AnCoRaGeM                2
ancoragem                2
Ancorajen                2
E

In [3]:
import unicodedata

def normalizar_texto(texto):
    # Converte para string, remove espaços e coloca em minúsculo
    texto = str(texto).strip().lower()
    # Remove acentos
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    # Remove espaços internos (resolve "E L E T R Ô N I C O S")
    texto = texto.replace(' ', '')
    return texto

def classificar_categoria(texto):
    t = normalizar_texto(texto)
    # Verifica qual categoria o texto normalizado pertence
    if 'eletron' in t or 'eletrun' in t:
        return 'eletrônicos'
    elif 'prop' in t:
        return 'propulsão'
    elif 'ancor' in t or 'encor' in t:
        return 'ancoragem'
    else:
        return 'desconhecido'

# Aplica a classificação em cada linha da coluna actual_category
df_produtos['category'] = df_produtos['actual_category'].apply(classificar_categoria)

# Verifica o resultado
print(df_produtos['category'].value_counts())

category
propulsão      53
ancoragem      53
eletrônicos    51
Name: count, dtype: int64


In [4]:
# Mostra quais categorias brutas viraram 'desconhecido' para investigar
print(df_produtos[df_produtos['category'] == 'desconhecido']['actual_category'].value_counts())

Series([], Name: count, dtype: int64)


In [5]:
print(df_produtos.head())
df_produtos.head()

                             name        price  code        actual_category  \
0     Transponder AIS Maré Magnum  R$ 33122.52     1            ELETRONICOS   
1       Transponder Furuno Marlin  R$ 13998.15     2            ELETRONICOS   
2    Radar Furuno Pulse Leviathan   R$ 9024.19     3  E L E T R Ô N I C O S   
3       Rádio AIS Hydro Tidal Zen   R$ 3381.88     4            Eletrunicos   
4  Piloto Automático Furuno Storm  R$ 23669.01     5            Eletronicoz   

      category  
0  eletrônicos  
1  eletrônicos  
2  eletrônicos  
3  eletrônicos  
4  eletrônicos  


,name,price,code,actual_category,category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS,eletrônicos
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS,eletrônicos
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S,eletrônicos
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos,eletrônicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz,eletrônicos


In [6]:
# PARTE 2: Converte a coluna price para tipo numérico
# errors='coerce' transforma valores inválidos em NaN em vez de dar erro
df_produtos['price'] = df_produtos['price'].str.replace('R$', '', regex=False).str.strip()
df_produtos['price'] = pd.to_numeric(df_produtos['price'], errors='coerce')

# Verifica o tipo da coluna e se há valores inválidos
print("Tipo da coluna price:", df_produtos['price'].dtype)
print("Valores nulos em price:", df_produtos['price'].isna().sum())
print("\nExemplos de valores convertidos:")
print(df_produtos['price'].head(5))

# PARTE 3: Remove duplicatas
total_antes = len(df_produtos)
df_produtos = df_produtos.drop_duplicates()
total_depois = len(df_produtos)

print(f"\nLinhas antes: {total_antes}")
print(f"Linhas depois: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

Tipo da coluna price: float64
Valores nulos em price: 0

Exemplos de valores convertidos:
0    33122.52
1    13998.15
2     9024.19
3     3381.88
4    23669.01
Name: price, dtype: float64

Linhas antes: 157
Linhas depois: 157
Duplicatas removidas: 0


In [7]:
# Investiga os valores brutos da coluna price antes da conversão
# Recarrega o CSV para ver os dados originais sem a transformação
df_raw = pd.read_csv(PRODUTOS)
print("Exemplos de valores em price:")
print(df_raw['price'].head(10))
print("\nTipo original:", df_raw['price'].dtype)

Exemplos de valores em price:
0    R$ 33122.52
1    R$ 13998.15
2     R$ 9024.19
3     R$ 3381.88
4    R$ 23669.01
5    R$ 11820.21
6    R$ 19518.77
7     R$ 4984.15
8     R$ 39705.5
9    R$ 32033.04
Name: price, dtype: str

Tipo original: str


In [ ]:
# Verifica os nomes reais das colunas do CSV de vendas
print(df_vendas.columns.tolist())

['id', 'id_client', 'id_product', 'qtd', 'total', 'sale_date']


In [10]:
# ── Q1 — EDA ──────────────────────────────────────────────────────
# Carrega o CSV de vendas sem nenhuma transformação
df_vendas = pd.read_csv(VENDAS)

# PARTE 1: Visão geral
print("Linhas:", len(df_vendas))
print("Colunas:", len(df_vendas.columns))
print("Nomes das colunas:", df_vendas.columns.tolist())

# Intervalo de datas
print("\nData mínima:", df_vendas['sale_date'].min())
print("Data máxima:", df_vendas['sale_date'].max())

# PARTE 2: Análise da coluna total
print("\nValor mínimo em total:", df_vendas['total'].min())
print("Valor máximo em total:", df_vendas['total'].max())
print("Valor médio em total:", round(df_vendas['total'].mean(), 2))

# Qualidade geral
print("\nValores nulos por coluna:")
print(df_vendas.isnull().sum())

Linhas: 9895
Colunas: 6
Nomes das colunas: ['id', 'id_client', 'id_product', 'qtd', 'total', 'sale_date']

Data mínima: 01-01-2023
Data máxima: 31-12-2024

Valor mínimo em total: 294.5
Valor máximo em total: 2222973.0
Valor médio em total: 263797.83

Valores nulos por coluna:
id            0
id_client     0
id_product    0
qtd           0
total         0
sale_date     0
dtype: int64


In [12]:

# Q1.1 — mesmas métricas via SQL com DuckDB
conn = duckdb.connect()

resultado = conn.execute("""
    SELECT
        COUNT(*)                        AS total_linhas,
        MIN(sale_date)                  AS data_minima,
        MAX(sale_date)                  AS data_maxima,
        MIN(total)                      AS valor_minimo,
        MAX(total)                      AS valor_maximo,
        ROUND(AVG(total), 2)            AS valor_medio
    FROM read_csv_auto('vendas_2023_2024.csv')
""").df()

print(resultado.to_string(index=False))

 total_linhas data_minima data_maxima  valor_minimo  valor_maximo  valor_medio
         9895  01-01-2023  31-12-2024         294.5     2222973.0    263797.83


In [13]:
import json

# Abre e inspeciona o JSON de custos
with open(CUSTOS) as f:
    custos_raw = json.load(f)

# Mostra a estrutura dos 2 primeiros registros
print(json.dumps(custos_raw[:2], indent=2))

[
  {
    "product_id": 1,
    "product_name": "Transponder AIS Mar\u00e9 Magnum",
    "category": "eletr\u00f4nicos",
    "historic_data": [
      {
        "start_date": "10/08/2016",
        "usd_price": 10583.63
      },
      {
        "start_date": "15/06/2018",
        "usd_price": 8778.36
      },
      {
        "start_date": "25/09/2018",
        "usd_price": 8023.87
      },
      {
        "start_date": "19/03/2019",
        "usd_price": 8772.78
      },
      {
        "start_date": "17/01/2020",
        "usd_price": 7918.18
      },
      {
        "start_date": "17/06/2020",
        "usd_price": 6310.01
      },
      {
        "start_date": "02/07/2021",
        "usd_price": 6586.7
      },
      {
        "start_date": "16/05/2022",
        "usd_price": 6538.2
      },
      {
        "start_date": "28/02/2023",
        "usd_price": 6360.91
      },
      {
        "start_date": "17/10/2023",
        "usd_price": 6574.8
      },
      {
        "start_date": "16/02/202

In [14]:
# Q3 — normaliza o JSON aninhado em linhas flat
linhas = []

for produto in custos_raw:
    for entrada in produto['historic_data']:
        linhas.append({
            'product_id':   produto['product_id'],
            'product_name': produto['product_name'],
            'category':     produto['category'],
            'start_date':   entrada['start_date'],
            'usd_price':    entrada['usd_price']
        })

df_custos = pd.DataFrame(linhas)

print("Total de entradas:", len(df_custos))
print(df_custos.head())

Total de entradas: 1260
   product_id                 product_name     category  start_date  usd_price
0           1  Transponder AIS Maré Magnum  eletrônicos  10/08/2016   10583.63
1           1  Transponder AIS Maré Magnum  eletrônicos  15/06/2018    8778.36
2           1  Transponder AIS Maré Magnum  eletrônicos  25/09/2018    8023.87
3           1  Transponder AIS Maré Magnum  eletrônicos  19/03/2019    8772.78
4           1  Transponder AIS Maré Magnum  eletrônicos  17/01/2020    7918.18


In [15]:
# Salva o resultado como CSV
df_custos.to_csv('custos_importacao_normalizado.csv', index=False)
print("Arquivo salvo!")

Arquivo salvo!


In [17]:
# Verifica o formato das datas em vendas e em custos
print("Exemplo sale_date:", df_vendas['sale_date'].iloc[0])
print("Exemplo start_date:", df_custos['start_date'].iloc[0])

Exemplo sale_date: 2023-09-10
Exemplo start_date: 10/08/2016


In [ ]:
pip install requests  # instala o requests na venv

Note: you may need to restart the kernel to use updated packages.


In [20]:
import requests

# Busca cotação do dólar comercial no Bacen para todo o período
# A API retorna a média da cotação de venda por dia
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@di,dataFinalCotacao=@df)"
    "?@di='01-01-2023'&@df='12-31-2024'"
    "&$format=json&$select=cotacaoVenda,dataHoraCotacao"
)

resp = requests.get(url)
cambio_raw = resp.json()['value']

print(f"Cotações recebidas: {len(cambio_raw)}")
print("Exemplo:", cambio_raw[0])

Cotações recebidas: 502
Exemplo: {'cotacaoVenda': 5.3436, 'dataHoraCotacao': '2023-01-02 13:05:57.593'}


In [25]:
# Transforma em DataFrame e extrai só a data (sem hora)
df_cambio = pd.DataFrame(cambio_raw)
df_cambio['data'] = pd.to_datetime(df_cambio['dataHoraCotacao']).dt.date.astype(str)
df_cambio = df_cambio[['data', 'cotacaoVenda']].rename(columns={'cotacaoVenda': 'taxa_cambio'})

# Mantém só a última cotação do dia (Bacen pode ter múltiplas por dia)
df_cambio = df_cambio.sort_values('data').drop_duplicates('data', keep='last')

print("Dias com cotação:", len(df_cambio))
print(df_cambio.head())

# Cria calendário completo e propaga cotação para fins de semana
datas_completas = pd.date_range('2023-01-01', '2024-12-31').astype(str)
df_cambio_full = pd.DataFrame({'data': datas_completas})

# Junta com as cotações reais
df_cambio_full = df_cambio_full.merge(df_cambio, on='data', how='left')

# Propaga a última cotação conhecida para dias sem cotação (fins de semana e feriados)
df_cambio_full['taxa_cambio'] = df_cambio_full['taxa_cambio'].ffill()

print("Total de dias:", len(df_cambio_full))
print("Dias sem cotação após ffill:", df_cambio_full['taxa_cambio'].isna().sum())
print(df_cambio_full.head(8))

# Preenche o 01/01 com a primeira cotação disponível (feriado sem anterior)
df_cambio_full['taxa_cambio'] = df_cambio_full['taxa_cambio'].bfill()

print("Dias sem cotação:", df_cambio_full['taxa_cambio'].isna().sum())

Dias com cotação: 502
         data  taxa_cambio
0  2023-01-02       5.3436
1  2023-01-03       5.3759
2  2023-01-04       5.4459
3  2023-01-05       5.4026
4  2023-01-06       5.2855
Total de dias: 731
Dias sem cotação após ffill: 1
         data  taxa_cambio
0  2023-01-01          NaN
1  2023-01-02       5.3436
2  2023-01-03       5.3759
3  2023-01-04       5.4459
4  2023-01-05       5.4026
5  2023-01-06       5.2855
6  2023-01-07       5.2855
7  2023-01-08       5.2855
Dias sem cotação: 0


In [26]:
# Junta vendas com câmbio pela data
df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date']).dt.date.astype(str)
df_q4 = df_vendas.merge(df_cambio_full, left_on='sale_date', right_on='data', how='left')

# Verifica
print("Vendas sem câmbio:", df_q4['taxa_cambio'].isna().sum())
print(df_q4[['id_product', 'sale_date', 'total', 'taxa_cambio']].head())

ValueError: time data "15-09-2024" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [27]:
# Junta vendas com câmbio pela data
df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date'], format='mixed', dayfirst=True).dt.date.astype(str)
df_q4 = df_vendas.merge(df_cambio_full, left_on='sale_date', right_on='data', how='left')

print("Vendas sem câmbio:", df_q4['taxa_cambio'].isna().sum())
print(df_q4[['id_product', 'sale_date', 'total', 'taxa_cambio']].head())

Vendas sem câmbio: 0
   id_product   sale_date     total  taxa_cambio
0         105  2023-10-09    3405.0       5.1666
1         136  2024-09-15   16873.9       5.5717
2         139  2024-08-13    9475.3       5.4875
3          23  2023-03-02   55893.0       5.2080
4          57  2024-12-02  451403.9       6.0634


In [29]:
# Converte datas para datetime para o merge_asof funcionar
df_q4['sale_date_dt'] = pd.to_datetime(df_q4['sale_date'])
df_custos_sorted['start_date_dt'] = pd.to_datetime(df_custos_sorted['start_date'])

df_q4 = pd.merge_asof(
    df_q4.sort_values('sale_date_dt'),
    df_custos_sorted[['product_id', 'start_date_dt', 'usd_price']].sort_values('start_date_dt'),
    left_on='sale_date_dt',
    right_on='start_date_dt',
    left_by='id_product',
    right_by='product_id',
    direction='backward'
)

print("Vendas sem custo USD:", df_q4['usd_price'].isna().sum())
print(df_q4[['id_product', 'sale_date', 'total', 'taxa_cambio', 'usd_price']].head())

Vendas sem custo USD: 0
   id_product   sale_date       total  taxa_cambio  usd_price
0          15  2023-01-01   132524.05       5.3436    5325.40
1         130  2023-01-01    53873.00       5.3436     749.89
2          78  2023-01-01  1461139.00       5.3436   23053.04
3          44  2023-01-01    51332.30       5.3436    1963.02
4          22  2023-01-01    74960.70       5.3436    7630.67


In [30]:
# Calcula custo total em BRL e identifica prejuízo por transação
df_q4['custo_brl'] = df_q4['usd_price'] * df_q4['taxa_cambio']
df_q4['custo_total_brl'] = df_q4['custo_brl'] * df_q4['qtd']
df_q4['prejuizo'] = (df_q4['custo_total_brl'] - df_q4['total']).clip(lower=0)

# Agrega por produto
df_resultado = df_q4.groupby('id_product').agg(
    receita_total=('total', 'sum'),
    prejuizo_total=('prejuizo', 'sum')
).reset_index()

df_resultado['pct_perda'] = (df_resultado['prejuizo_total'] / df_resultado['receita_total'] * 100).round(2)

# Filtra só produtos com prejuízo
df_prejuizo = df_resultado[df_resultado['prejuizo_total'] > 0].sort_values('prejuizo_total', ascending=False)

print(df_prejuizo.head(10))

    id_product  receita_total  prejuizo_total  pct_perda
71          72    63057815.65    3.995969e+07      63.37
82          83    44377440.00    1.838238e+07      41.42
73          74    59764356.15    7.004602e+06      11.72
70          71    81567066.65    6.120380e+06       7.50
54          55    61224375.00    5.597894e+06       9.14
90          91    66829268.70    5.446134e+06       8.15
62          63    68817185.90    4.614028e+06       6.70
77          78    58043733.35    4.518044e+06       7.78
59          60    53418532.00    3.035103e+06       5.68
95          96    27227868.90    3.001834e+06      11.02


In [31]:
# Q4 respostas objetivas
maior_prejuizo_abs = df_prejuizo.iloc[0]
maior_pct_perda = df_prejuizo.sort_values('pct_perda', ascending=False).iloc[0]

print("Produto com MAIOR prejuízo absoluto:")
print(f"  id_product: {maior_prejuizo_abs['id_product']}  |  prejuízo: R$ {maior_prejuizo_abs['prejuizo_total']:,.2f}")

print("\nProduto com MAIOR % de perda:")
print(f"  id_product: {maior_pct_perda['id_product']}  |  % perda: {maior_pct_perda['pct_perda']}%")

print("\nSão o mesmo produto?", maior_prejuizo_abs['id_product'] == maior_pct_perda['id_product'])

Produto com MAIOR prejuízo absoluto:
  id_product: 72.0  |  prejuízo: R$ 39,959,694.85

Produto com MAIOR % de perda:
  id_product: 72.0  |  % perda: 63.37%

São o mesmo produto? True


In [32]:
# Q4.1 — SQL equivalente via DuckDB
resultado_sql = conn.execute("""
WITH cambio AS (
    -- Lê cotações do Bacen já salvas como CSV
    SELECT data, taxa_cambio
    FROM df_cambio_full
),
vendas_cambio AS (
    SELECT
        v.id_product,
        v.sale_date,
        v.total,
        v.qtd,
        c.taxa_cambio
    FROM df_vendas v
    LEFT JOIN cambio c ON v.sale_date = c.data
),
com_custo AS (
    SELECT
        vc.*,
        cu.usd_price,
        cu.usd_price * vc.taxa_cambio * vc.qtd AS custo_total_brl,
        GREATEST(cu.usd_price * vc.taxa_cambio * vc.qtd - vc.total, 0) AS prejuizo
    FROM vendas_cambio vc
    LEFT JOIN df_custos cu ON vc.id_product = cu.product_id
        AND cu.start_date = (
            SELECT MAX(start_date) FROM df_custos
            WHERE product_id = vc.id_product
            AND start_date <= vc.sale_date
        )
)
SELECT
    id_product,
    ROUND(SUM(total), 2)       AS receita_total,
    ROUND(SUM(prejuizo), 2)    AS prejuizo_total,
    ROUND(SUM(prejuizo) / SUM(total) * 100, 2) AS pct_perda
FROM com_custo
WHERE prejuizo > 0
GROUP BY id_product
ORDER BY prejuizo_total DESC
""").df()

print(resultado_sql.head(10))

   id_product  receita_total  prejuizo_total  pct_perda
0          72    63057815.65     39959694.85      63.37
1          83    44377440.00     18382383.33      41.42
2          74    45183566.35      7004602.39      15.50
3          71    70372097.65      6120379.87       8.70
4          55    59574567.00      5597894.25       9.40
5          91    56955613.50      5446133.67       9.56
6          63    53099941.25      4614028.37       8.69
7          78    51614723.25      4518043.60       8.75
8          60    39150389.05      3035102.62       7.75
9          96    24441876.20      3001834.01      12.28


## Q5

In [33]:
# Junta vendas com produtos limpos para ter a categoria por transação
df_q5 = df_vendas.merge(
    df_produtos[['code', 'category']],
    left_on='id_product',
    right_on='code',
    how='left'
)

print("Nulos em category:", df_q5['category'].isna().sum())
print(df_q5[['id_client', 'id_product', 'total', 'category']].head())

Nulos em category: 0
   id_client  id_product     total     category
0         42         105    3405.0    ancoragem
1          3         136   16873.9    ancoragem
2         25         139    9475.3    ancoragem
3         20          23   55893.0  eletrônicos
4          8          57  451403.9    propulsão


In [34]:
# Calcula faturamento, frequência, ticket médio e diversidade de categorias
df_clientes = df_q5.groupby('id_client').agg(
    faturamento_total=('total', 'sum'),
    frequencia=('id', 'count'),
    diversidade=('category', 'nunique')
).reset_index()

df_clientes['ticket_medio'] = df_clientes['faturamento_total'] / df_clientes['frequencia']

# Filtra só clientes com 3+ categorias distintas
df_elite = df_clientes[df_clientes['diversidade'] >= 3]

# Top 10 por ticket médio, desempate por id_client crescente
df_top10 = df_elite.sort_values(['ticket_medio', 'id_client'], ascending=[False, True]).head(10)

print(df_top10[['id_client', 'ticket_medio', 'diversidade', 'frequencia']].to_string(index=False))

 id_client  ticket_medio  diversidade  frequencia
        47 334991.747772            3         202
        42 319059.596336            3         232
        46 311805.458867            3         203
         9 310672.413889            3         234
        36 301521.448696            3         230
         2 300964.690000            3         225
        22 298318.472000            3         200
        28 298268.631136            3         220
        26 297558.483036            3         224
         5 290815.536667            3         210


In [35]:
# Filtra vendas só dos top 10 clientes
top10_ids = df_top10['id_client'].tolist()
df_top10_vendas = df_q5[df_q5['id_client'].isin(top10_ids)]

# Categoria com maior soma de qtd
categoria_top = df_top10_vendas.groupby('category')['qtd'].sum().sort_values(ascending=False)
print(categoria_top)

category
propulsão      6429
ancoragem      6003
eletrônicos    5376
Name: qtd, dtype: int64


In [36]:
# Q6 — Dimensão de datas com DuckDB
resultado_q6 = conn.execute("""
WITH calendario AS (
    -- Gera todas as datas do período
    SELECT UNNEST(generate_series(
        MIN(sale_date)::DATE,
        MAX(sale_date)::DATE,
        INTERVAL '1 day'
    )) AS data
    FROM df_vendas
),
vendas_dia AS (
    SELECT sale_date::DATE AS data, SUM(total) AS valor_venda
    FROM df_vendas
    GROUP BY sale_date::DATE
),
calendario_vendas AS (
    SELECT
        c.data,
        COALESCE(v.valor_venda, 0) AS valor_venda,
        CASE DAYOFWEEK(c.data)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado'
        END AS dia_semana
    FROM calendario c
    LEFT JOIN vendas_dia v ON c.data = v.data
)
SELECT
    dia_semana,
    ROUND(AVG(valor_venda), 2) AS media_vendas
FROM calendario_vendas
GROUP BY dia_semana
ORDER BY media_vendas ASC
""").df()

print(resultado_q6)

      dia_semana  media_vendas
0        Domingo    3229614.16
1  Segunda-feira    3484500.47
2    Terça-feira    3488871.99
3   Quarta-feira    3534007.21
4   Quinta-feira    3713299.94
5         Sábado    3774290.79
6    Sexta-feira    3776151.25


In [37]:
# Filtra o produto alvo
produto_alvo = "Motor de Popa Yamaha Evo Dash 155HP"

# Busca o id do produto
id_alvo = df_produtos[df_produtos['name'] == produto_alvo]['code'].values[0]
print("ID do produto:", id_alvo)

# Agrupa vendas diárias do produto
df_motor = df_q5[df_q5['id_product'] == id_alvo][['sale_date', 'qtd']].copy()
df_motor['sale_date'] = pd.to_datetime(df_motor['sale_date'])
df_motor = df_motor.groupby('sale_date')['qtd'].sum().reset_index()

print(f"Registros de venda: {len(df_motor)}")
print(df_motor.head())

ID do produto: 54
Registros de venda: 58
   sale_date  qtd
0 2023-01-10    3
1 2023-02-06   13
2 2023-02-27   15
3 2023-03-04   14
4 2023-03-15    4


In [38]:
# Cria calendário completo e preenche dias sem venda com 0
datas = pd.date_range('2023-01-01', '2024-12-31')
df_motor_full = pd.DataFrame({'sale_date': datas})
df_motor_full = df_motor_full.merge(df_motor, on='sale_date', how='left')
df_motor_full['qtd'] = df_motor_full['qtd'].fillna(0)

# Separa treino e teste
treino = df_motor_full[df_motor_full['sale_date'] <= '2023-12-31'].copy()
teste  = df_motor_full[(df_motor_full['sale_date'] >= '2024-01-01') &
                       (df_motor_full['sale_date'] <= '2024-01-31')].copy()

# Média móvel 7 dias — só dados anteriores à data prevista (sem data leakage)
historico = treino['qtd'].tolist()
previsoes = []

for i in range(len(teste)):
    janela = historico[-7:]
    previsoes.append(round(sum(janela) / len(janela), 4))
    historico.append(teste['qtd'].iloc[i])  # adiciona real ao histórico após prever

teste['previsao'] = previsoes

# MAE
mae = (teste['qtd'] - teste['previsao']).abs().mean()
print(f"MAE: {mae:.4f}")
print(f"\nSoma previsão 01/01 a 07/01: {round(teste[teste['sale_date'] <= '2024-01-07']['previsao'].sum())}")
print(teste[['sale_date', 'qtd', 'previsao']].head(10))

MAE: 1.6406

Soma previsão 01/01 a 07/01: 3
     sale_date   qtd  previsao
365 2024-01-01   0.0    0.0000
366 2024-01-02   0.0    0.0000
367 2024-01-03   0.0    0.0000
368 2024-01-04   0.0    0.0000
369 2024-01-05  10.0    0.0000
370 2024-01-06   0.0    1.4286
371 2024-01-07   0.0    1.4286
372 2024-01-08   0.0    1.4286
373 2024-01-09   0.0    1.4286
374 2024-01-10   0.0    1.4286


## Q8

In [39]:
import numpy as np

# Matriz usuário-produto (1 se comprou, 0 se não)
df_matrix = df_vendas.groupby(['id_client', 'id_product'])['qtd'].sum().reset_index()
df_matrix['comprou'] = 1
matriz = df_matrix.pivot_table(index='id_client', columns='id_product', values='comprou', fill_value=0)

# ID do GPS Garmin Vortex Maré Drift
id_gps = df_produtos[df_produtos['name'] == 'GPS Garmin Vortex Maré Drift']['code'].values[0]
print("ID do GPS:", id_gps)

# Similaridade de cosseno entre produtos
M = matriz.values.T  # produtos nas linhas
normas = np.linalg.norm(M, axis=1, keepdims=True)
normas[normas == 0] = 1  # evita divisão por zero
M_norm = M / normas
sim_matrix = M_norm @ M_norm.T

# Ranking dos 5 mais similares ao GPS
produtos_ids = matriz.columns.tolist()
idx_gps = produtos_ids.index(id_gps)
similares = sim_matrix[idx_gps]

ranking = sorted(enumerate(similares), key=lambda x: x[1], reverse=True)
ranking = [(produtos_ids[i], s) for i, s in ranking if produtos_ids[i] != id_gps][:5]

print("\nTop 5 produtos similares ao GPS:")
for pid, sim in ranking:
    nome = df_produtos[df_produtos['code'] == pid]['name'].values
    print(f"  id: {pid}  |  sim: {sim:.4f}  |  {nome[0] if len(nome) else 'N/A'}")

ID do GPS: 27

Top 5 produtos similares ao GPS:
  id: 94  |  sim: 0.8696  |  Motor de Popa Volvo Magnum 276HP
  id: 11  |  sim: 0.8680  |  GPS Furuno Swift Leviathan Poseidon
  id: 35  |  sim: 0.8539  |  Radar Furuno Swift
  id: 1  |  sim: 0.8500  |  Transponder AIS Maré Magnum
  id: 115  |  sim: 0.8500  |  Cabo de Nylon Delta Force Magnum Leviathan
